FileNotFoundError: [Errno 2] No such file or directory: 'twitter_training.csv'

,content,content_nettoye
0,im getting on borderlands and i will murder yo...,im getting borderland murder
1,I am coming to the borders and I will kill you...,coming border kill
2,im getting on borderlands and i will kill you ...,im getting borderland kill
3,im coming on borderlands and i will murder you...,im coming borderland murder
4,im getting on borderlands 2 and i will murder ...,im getting borderland 2 murder
...,...,...
995,of,
996,Who's down for some @Borderlands on,who borderland
997,Who's on for some @ Borderlands,who borderland
998,Who's at @ Borderlands,who borderland


['02' '0sku6vr4ixu' '10' ... 'youve' 'zelda' 'zer0']
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Dimensions de la matrice : (1000, 1912)


[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
['02' '0sku6vr4ixu' '10' ... 'youve' 'zelda' 'zer0']
Dimensions de la matrice de caractéristiques : (1000, 1912)


In [ ]:
import numpy as np
import pandas as pd
import re
import string
import math
from collections import Counter
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# =====================================================================
# 1. CHARGEMENT DU JEU DE DONNÉES
# =====================================================================
chemin = 'twitter_training.csv'
df = pd.read_csv(chemin, header=None, names=['id', 'entite', 'sentiment', 'contenu'])

# Sélection des 1000 premières lignes et nettoyage des valeurs manquantes
df_selection = df.head(1000).copy()
df_selection = df_selection.dropna(subset=['contenu'])


# =====================================================================
# 2. PHASE DE NETTOYAGE (PRÉTRAITEMENT)
# =====================================================================
mots_vides = set(stopwords.words('english'))
lemmatiseur = WordNetLemmatizer()

def preotraitement(texte):
    if not isinstance(texte, str):
        return ""
    texte = re.sub(r"[^\w\s]", "", texte)
    texte = texte.lower()
    tokens = word_tokenize(texte)
    tokens = [
        lemmatiseur.lemmatize(mot)
        for mot in tokens
        if mot not in mots_vides and mot not in string.punctuation
    ]
    return ' '.join(tokens)

# Création de la colonne nettoyée AVANT le TF-IDF
df_selection['contenu_nettoye'] = df_selection['contenu'].apply(preotraitement)


# =====================================================================
# 3. TRANSFORMATION NUMÉRIQUE (TF-IDF PERSONNALISÉ)
# =====================================================================
class SimulationMatriceCreuse:
    """Classe utilitaire pour simuler le comportement de scikit-learn"""
    def __init__(self, donnees):
        self.data = donnees
        self.shape = donnees.shape
    def toarray(self):
        return self.data

class TFIDFPersonnalise:
    def __init__(self):
        self.vocabulaire = {}
        self.vecteur_idf = {}

    def _tokeniser(self, texte):
        return re.findall(r"(?u)\b\w\w+\b", str(texte).lower())

    def fit(self, corpus):
        nbre_documents = len(corpus)
        comptages_df = Counter()
        for doc in corpus:
            tokens = set(self._tokeniser(doc))
            for token in tokens:
                comptages_df[token] += 1
        mots_tries = sorted(comptages_df.keys())
        self.vocabulaire = {mot: i for i, mot in enumerate(mots_tries)}
        for mot in self.vocabulaire:
            compte = comptages_df[mot]
            self.vecteur_idf[mot] = math.log((1 + nbre_documents) / (1 + compte)) + 1
        return self

    def transform(self, corpus):
        lignes = []
        taille_vocabulaire = len(self.vocabulaire)
        for doc in corpus:
            tokens = self._tokeniser(doc)
            comptages_tf = Counter(tokens)
            vecteur = np.zeros(taille_vocabulaire)
            for mot, compte in comptages_tf.items():
                if mot in self.vocabulaire:
                    vecteur[self.vocabulaire[mot]] = compte * self.vecteur_idf[mot]
            norme = np.linalg.norm(vecteur)
            if norme > 0:
                vecteur = vecteur / norme
            lignes.append(vecteur)
        return SimulationMatriceCreuse(np.array(lignes))

    def fit_transform(self, corpus):
        return self.fit(corpus).transform(corpus)

    def get_feature_names_out(self):
        return np.array([mot for mot, index in sorted(self.vocabulaire.items(), key=lambda item: item[1])])

# Application du TF-IDF
moteur_tfidf = TFIDFPersonnalise()
objet_caracteristiques = moteur_tfidf.fit_transform(df_selection['contenu_nettoye'])
matrice_caracteristiques = objet_caracteristiques.toarray()


# =====================================================================
# 4. PRÉPARATION DU FILTRAGE BINAIRE (POSITIF VS NÉGATIF)
# =====================================================================
masque_binaire = df_selection['sentiment'].isin(['Positive', 'Negative'])

X_final = matrice_caracteristiques[masque_binaire]
y_final = df_selection['sentiment'][masque_binaire].map({'Positive': 1, 'Negative': 0}).values

print(f"Dimensions de la matrice X_final : {X_final.shape}")


# =====================================================================
# 5. CODE PERSONNALISÉ DE LA RÉGRESSION LOGISTIQUE
# =====================================================================
class RegressionLogistiquePersonnalisee:
    def __init__(self, taux_apprentissage=0.5, nbre_iterations=400):
        self.taux_apprentissage = taux_apprentissage
        self.nbre_iterations = nbre_iterations
        self.poids = None
        self.biais = None
        self.historique_perte = []

    def _sigmoide(self, z):
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def _calculer_perte(self, y_vrai, y_pred):
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_vrai * np.log(y_pred) + (1 - y_vrai) * np.log(1 - y_pred))

    def fit(self, X, y):
        nbre_echantillons, nbre_caracteristiques = X.shape
        self.poids = np.zeros(nbre_caracteristiques)
        self.biais = 0.0
        self.historique_perte = []

        for i in range(self.nbre_iterations):
            modele_lineaire = np.dot(X, self.poids) + self.biais
            y_predit = self._sigmoide(modele_lineaire)

            perte = self._calculer_perte(y, y_predit)
            self.historique_perte.append(perte)

            # Gradients
            dpoids = (1 / nbre_echantillons) * np.dot(X.T, (y_predit - y))
            dbiais = (1 / nbre_echantillons) * np.sum(y_predit - y)

            # Mise à jour des paramètres
            self.poids -= self.taux_apprentissage * dpoids
            self.biais -= self.taux_apprentissage * dbiais

            if i % 50 == 0:
                print(f"Itération {i:3d} -> Perte (Loss): {perte:.4f}")
        return self

    def predict(self, X):
        modele_lineaire = np.dot(X, self.poids) + self.biais
        return (self._sigmoide(modele_lineaire) >= 0.5).astype(int)

# Entraînement du modèle
classifieur = RegressionLogistiquePersonnalisee(taux_apprentissage=0.6, nbre_iterations=400)
classifieur.fit(X_final, y_final)


# =====================================================================
# 6. GRAPHIQUE DE LA COURBE D'APPRENTISSAGE
# =====================================================================
plt.figure(figsize=(9, 5))
plt.plot(classifieur.historique_perte, color='darkorange', lw=2, label='Log Loss (Entropie Croisée)')
plt.title("Courbe d'apprentissage : Évolution de la Perte (Loss)", fontsize=12)
plt.xlabel("Itérations")
plt.ylabel("Perte (Loss)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# Évaluation de la précision exacte (Accuracy)
predictions = classifieur.predict(X_final)
precision = np.mean(predictions == y_final)
print(f"Précision finale sur l'échantillon : {precision * 100:.2f}%")

Dimensions de la matrice X_final : (615, 1912)
Itération   0 -> Perte (Loss): 0.6931
Itération  50 -> Perte (Loss): 0.5762
